In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install faiss-cpu

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print("Creating knowledge base")

kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base successfully created")

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

##Q1 Answer  0.384
result = zs(prompt_150, labels_150)
correct_idx = result['labels'].index(ans_150)
score = result['scores'][correct_idx]
print(round(score, 3))

In [ ]:
#Q2 Answer 10
query_embedding = model.encode([prompt_150])
distances, indices = index.search(query_embedding, k=10)
print("Retrieved indices:", indices[0])
rank = list(indices[0]).index(150) + 1
print(f"Rank of correct document: {rank}")

In [ ]:
#Q3 Answer 1
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
retrieved_indices = indices[0]
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)
ranked = sorted(zip(ce_scores, retrieved_indices), reverse=True)
print("Reranked indices:", [idx for _, idx in ranked])
rank = [idx for _, idx in ranked].index(150) + 1
print(f"Rank of correct document after reranking: {rank}")

In [ ]:
#Q4 Answer 216
from transformers import BertTokenizer

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
query_embedding_42 = model.encode([prompt_42])
distances_42, indices_42 = index.search(query_embedding_42, k=5)
docs_5 = [kb[i] for i in indices_42[0]]
concatenated = ' '.join(docs_5)
input_string = f"Context: {concatenated} Question: {prompt_42}"
tokens = bert_tokenizer(input_string, truncation=False)
print(f"Total tokens: {len(tokens['input_ids'])}")

In [ ]:
#Q5 Answer 0.989
true_doc = kb[150]
rag_string = f"Context: {true_doc} Question: {prompt_150}"
result_rag = zs(rag_string, labels_150)
correct_idx = result_rag['labels'].index(ans_150)
score_rag = result_rag['scores'][correct_idx]
print(round(score_rag, 3))

In [ ]:
#Q6 Answer 0.529
adversarial_doc = kb[999]
adversarial_string = f"Context: {adversarial_doc} Question: {prompt_150}"
result_adversarial = zs(adversarial_string, labels_150)
correct_idx = result_adversarial['labels'].index(ans_150)
score_adversarial = result_adversarial['scores'][correct_idx]
print(round(score_adversarial, 3))

In [ ]:
#Q7 Answer 73.0 
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_doc = str(row[row['answer']])
    query_emb = model.encode([prompt_i])
    _, retrieved = index.search(query_emb, k=5)
    docs = [kb[idx] for idx in retrieved[0]]
    if any(correct_doc in doc for doc in docs):
        hits += 1
hit_rate = (hits / 100) * 100
print(round(hit_rate, 1))

In [ ]:
#Q8 Answer 0.975
options_cols = ['A', 'B', 'C', 'D', 'E']
scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    labels_i = [str(row[col]) for col in options_cols]
    correct_i = str(row['answer'])

    # Retrieve
    query_emb = model.encode([prompt_i])
    _, retrieved = index.search(query_emb, k=5)
    docs_5 = [kb[idx] for idx in retrieved[0]]

    # Rerank
    pairs = [[prompt_i, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores)]

    # Predict
    result = zs(f"Context: {best_doc} Question: {prompt_i}", labels_i)

    # MAP@3 inline
    label_to_letter = {str(row[col]): col for col in options_cols}
    top3 = [label_to_letter.get(l) for l in result['labels'][:3]]
    score = next((1.0/(k+1) for k, l in enumerate(top3) if l == correct_i), 0.0)
    scores.append(score)

print(round(np.mean(scores), 3))